#  Planificador de viajes personalizado mediante RAG y LLM Customization

¿De qué se trata este notebook? Armamos un planificador de viajes personalizado que combina RAG (Retrieval-Augmented Generation) con un LLM open source. La idea de fondo: en vez de dejar que el modelo invente lugares "de memoria" (con el riesgo de que alucine restaurantes o direcciones que no existen), lo conectamos a una base de datos real de lugares en Buenos Aires, Nueva York y París, y le pedimos que arme itinerarios usando solo esa información.

A lo largo del notebook vamos a: cargar y unificar datos abiertos de las 3 ciudades → indexarlos en una base vectorial → construir el sistema de recuperación (RAG) → cargar el LLM que arma el itinerario final → y cerrar con una interfaz simple en Streamlit para probarlo de punta a punta.

## Configuración del entorno

Antes de arrancar con la parte interesante, dejamos todo el entorno listo: instalamos las librerías que vamos a necesitar (sentence-transformers, chromadb, streamlit) y montamos Google Drive, para que los datos, los embeddings y el modelo persistan entre sesiones de Colab.

In [1]:
!pip install -q -U sentence-transformers chromadb

In [2]:
!pip install -q streamlit

In [3]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [4]:
from google.colab import drive

drive.mount('/content/drive')

print("Google Drive montado correctamente")

Mounted at /content/drive
Google Drive montado correctamente


In [5]:
BASE_PATH = "/content/drive/MyDrive/travel_planner_rag"

CARPETAS = [
    "data",
    "chroma_db",
    "models",
    "outputs"
]

for carpeta in CARPETAS:
    os.makedirs(os.path.join(BASE_PATH, carpeta), exist_ok=True)

## Carga de Datasets

En esta etapa se cargan las fuentes de información utilizadas para construir
la base de conocimiento del sistema RAG.

Las fuentes contienen información turística como ciudades, categorías,
nombres de lugares, descripciones y ubicaciones.

Los datos son posteriormente procesados y unificados para generar un único
dataset que será utilizado como base documental del sistema de recuperación.

### Dataset multi-ciudad: Buenos Aires, Nueva York y París

Usamos datasets oficiales de portales de datos abiertos municipales, cubriendo los ejes
"cultura" y "gastronomía" para cada ciudad:

- Buenos Aires: data.buenosaires.gob.ar (CKAN, CSV)
- Nueva York: data.cityofnewyork.us (Socrata, CSV)
- París: opendata.paris.fr (Opendatasoft, CSV)

In [6]:
import pandas as pd

def leer_csv(url, sep_forzado=None, encoding="utf-8", **kwargs):
    try:
        if sep_forzado:
            df = pd.read_csv(url, sep=sep_forzado, encoding=encoding, **kwargs)
        else:
            df = pd.read_csv(url, sep=None, engine="python", encoding=encoding, **kwargs)
        return df if df.shape[0] > 0 else None
    except Exception as e:
        print(f"❌ Error leyendo {url}: {e}")
        return None

# --- Buenos Aires ---
url_ba_cultura = "https://data.buenosaires.gob.ar/dataset/monumentos/resource/juqdkmgo-1451-resource/download"
url_ba_gastronomia = "https://data.buenosaires.gob.ar/dataset/oferta-establecimientos-gastronomicos/resource/4dc39a90-07a4-4173-8a20-eb353a577a9f/download"

df_ba_cultura = leer_csv(url_ba_cultura, sep_forzado=";")
df_ba_gastronomia = leer_csv(url_ba_gastronomia, sep_forzado=";", encoding="latin-1")

# --- Nueva York ---
url_ny_cultura = "https://data.cityofnewyork.us/resource/pfja-tk2j.csv?$limit=5000"
url_ny_gastronomia = "https://data.cityofnewyork.us/resource/f6tk-2b7a.csv?$limit=5000"

df_ny_cultura = leer_csv(url_ny_cultura, sep_forzado=",")
df_ny_gastronomia = leer_csv(url_ny_gastronomia, sep_forzado=",")

# --- París ---
url_paris_cultura = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/que-faire-a-paris-/exports/csv"
url_paris_gastronomia = "https://opendata.paris.fr/explore/dataset/restaurants-casvp/download/?format=csv"

df_paris_cultura = leer_csv(url_paris_cultura, sep_forzado=";")
df_paris_gastronomia = leer_csv(url_paris_gastronomia, sep_forzado=";")

# --- Resumen final ---
datasets = {
    "BA - Cultura": df_ba_cultura,
    "BA - Gastronomía": df_ba_gastronomia,
    "NY - Cultura": df_ny_cultura,
    "NY - Gastronomía": df_ny_gastronomia,
    "Paris - Cultura": df_paris_cultura,
    "Paris - Gastronomía": df_paris_gastronomia,
}

for nombre, df in datasets.items():
    if df is not None:
        print(f"{nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")
    else:
        print(f"{nombre}: FALLÓ")

BA - Cultura: 2233 filas, 17 columnas
BA - Gastronomía: 124 filas, 11 columnas
NY - Cultura: 73 filas, 10 columnas
NY - Gastronomía: 5000 filas, 9 columnas
Paris - Cultura: 2843 filas, 69 columnas
Paris - Gastronomía: 43 filas, 6 columnas


In [7]:
carpeta_data = os.path.join(BASE_PATH, "data")

df_ba_cultura.to_csv(f"{carpeta_data}/ba_cultura.csv", index=False)
df_ba_gastronomia.to_csv(f"{carpeta_data}/ba_gastronomia.csv", index=False)
df_ny_cultura.to_csv(f"{carpeta_data}/ny_cultura.csv", index=False)
df_ny_gastronomia.to_csv(f"{carpeta_data}/ny_gastronomia.csv", index=False)
df_paris_cultura.to_csv(f"{carpeta_data}/paris_cultura.csv", index=False)
df_paris_gastronomia.to_csv(f"{carpeta_data}/paris_gastronomia.csv", index=False)

print("Los 6 datasets se guardaron en:", carpeta_data)

Los 6 datasets se guardaron en: /content/drive/MyDrive/travel_planner_rag/data


### Preprocesamiento: exploración inicial

Antes de limpiar, exploramos la estructura real de cada dataset: columnas, tipos de datos, valores faltantes. Esto determina qué columnas vamos a conservar para el RAG.

In [8]:
# Se verifican estrctura de los dataset y valores nulos
for nombre, df in datasets.items():
    print(f"===== {nombre} =====")
    print("Columnas:", df.columns.tolist())
    print("Valores nulos por columna:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print()

===== BA - Cultura =====
Columnas: ['ID', 'OBJETO_OBRA', 'CANTIDAD_OBJETOS', 'MATERIAL', 'DENOMINACION_SIMBOLIZA', 'AUTORES', 'UBICACION', 'OBSERVACIONES', 'DIRECCION_NORMALIZADA', 'CALLE', 'ALTURA', 'BARRIO', 'COMUNA', 'CODIGA_POSTAL', 'CODIGO_POSTAL_ARGENTINO', 'LATITUD', 'LONGITUD']
Valores nulos por columna:
MATERIAL                    12
DENOMINACION_SIMBOLIZA       1
AUTORES                    913
UBICACION                   17
OBSERVACIONES               17
DIRECCION_NORMALIZADA      119
CALLE                      119
ALTURA                     720
BARRIO                     312
COMUNA                     120
CODIGA_POSTAL              729
CODIGO_POSTAL_ARGENTINO    729
LATITUD                    130
LONGITUD                   130
dtype: int64

===== BA - Gastronomía =====
Columnas: ['nro_registro', 'rubro', 'establecimiento', 'domicilio_completo', 'barrio', 'comuna', 'telefono', 'email', 'facebook', 'web', 'accesibilidad']
Valores nulos por columna:
email              2
accesib

In [9]:
# Transformamos cada uno de los 6 datasets a una estructura común: {ciudad, categoria, nombre, descripcion, ubicacion}

def limpiar_texto(serie):
    return serie.astype(str).str.strip().replace({"nan": None, "": None})

# --- BA - Cultura ---
# Limpiamos los valores nulos (NaN) de autor/material/dirección/barrio en el dataset de Buenos Aires antes de convertirlos a texto, para que no aparecieran como el string literal "nan"
def valor_o_vacio(serie):
    return serie.where(serie.notna(), "")

autores = valor_o_vacio(df_ba_cultura["AUTORES"])
materiales = valor_o_vacio(df_ba_cultura["MATERIAL"])

descripcion_ba = ("Autor: " + autores + " | Material: " + materiales)
descripcion_ba = descripcion_ba.str.replace(r"^Autor:\s*\|\s*", "", regex=True)  # si no hay autor, sacamos el prefijo entero
descripcion_ba = descripcion_ba.str.strip(" |")

direccion = valor_o_vacio(df_ba_cultura["DIRECCION_NORMALIZADA"])
direccion = direccion.where(direccion != "", valor_o_vacio(df_ba_cultura["UBICACION"]))
barrio = valor_o_vacio(df_ba_cultura["BARRIO"])

ubicacion_ba = (direccion + ", " + barrio).str.strip(", ").str.replace(r",\s*,", ",", regex=True)

ba_cultura = pd.DataFrame({
    "ciudad": "Buenos Aires",
    "categoria": limpiar_texto(df_ba_cultura["OBJETO_OBRA"]),
    "nombre": limpiar_texto(df_ba_cultura["DENOMINACION_SIMBOLIZA"]),
    "descripcion": descripcion_ba,
    "ubicacion": ubicacion_ba,
})

# --- BA - Gastronomía ---
ba_gastronomia = pd.DataFrame({
    "ciudad": "Buenos Aires",
    "categoria": limpiar_texto(df_ba_gastronomia["rubro"]),
    "nombre": limpiar_texto(df_ba_gastronomia["establecimiento"]),
    "descripcion": limpiar_texto(df_ba_gastronomia["rubro"]) + " en el barrio de " + limpiar_texto(df_ba_gastronomia["barrio"]).fillna(""),
    "ubicacion": limpiar_texto(df_ba_gastronomia["domicilio_completo"]),
})

# --- NY - Cultura ---
ny_cultura = pd.DataFrame({
    "ciudad": "New York",
    "categoria": limpiar_texto(df_ny_cultura["discipline"]),
    "nombre": limpiar_texto(df_ny_cultura["organization_name"]),
    "descripcion": limpiar_texto(df_ny_cultura["discipline"]).fillna("Cultural organization") + " - " + limpiar_texto(df_ny_cultura["borough"]).fillna(""),
    "ubicacion": limpiar_texto(df_ny_cultura["address"]),
})

# --- NY - Gastronomía (deduplicar por restaurante, descartar columnas de inspección) ---
df_ny_gastro_dedup = df_ny_gastronomia.drop_duplicates(subset=["camis"])
ny_gastronomia = pd.DataFrame({
    "ciudad": "New York",
    "categoria": "Restaurant",  # el dataset no trae tipo de cocina, lo documentamos como limitación
    "nombre": limpiar_texto(df_ny_gastro_dedup["dba"]),
    "descripcion": "Restaurant located in " + limpiar_texto(df_ny_gastro_dedup["boro"]).fillna(""),
    "ubicacion": limpiar_texto(df_ny_gastro_dedup["street"]) + ", " + limpiar_texto(df_ny_gastro_dedup["boro"]).fillna(""),
})

# --- Paris - Cultura (dataset de eventos, tomamos title/description/address) ---
palabras_clave_turisticas = ["Histoire", "Expo", "Balade", "Patrimoine"] # nos quedamos solo con estos eventos, el resto son temporales

mask_turistico = df_paris_cultura["qfap_tags"].astype(str).str.contains(
    "|".join(palabras_clave_turisticas), case=False, na=False
)
df_paris_cultura_filtrado = df_paris_cultura[mask_turistico]

paris_cultura = pd.DataFrame({
    "ciudad": "Paris",
    "categoria": limpiar_texto(df_paris_cultura_filtrado["qfap_tags"]).fillna("Cultura"),
    "nombre": limpiar_texto(df_paris_cultura_filtrado["title"]),
    "descripcion": limpiar_texto(df_paris_cultura_filtrado["lead_text"]).fillna(limpiar_texto(df_paris_cultura_filtrado["description"])).astype(str).str.slice(0, 200),
    "ubicacion": limpiar_texto(df_paris_cultura_filtrado["address_name"]).fillna("") + ", " + limpiar_texto(df_paris_cultura_filtrado["address_street"]).fillna(""),
})

# --- Paris - Gastronomía ---
paris_gastronomia = pd.DataFrame({
    "ciudad": "Paris",
    "categoria": limpiar_texto(df_paris_gastronomia["type"]),
    "nombre": limpiar_texto(df_paris_gastronomia["nom_restaurant"]),
    "descripcion": limpiar_texto(df_paris_gastronomia["type"]).fillna("Restaurant"),
    "ubicacion": limpiar_texto(df_paris_gastronomia["adresse"]),
})

# --- Unificar todo ---
dataset_unificado = pd.concat([
    ba_cultura, ba_gastronomia,
    ny_cultura, ny_gastronomia,
    paris_cultura, paris_gastronomia,
], ignore_index=True)

# Limpieza final: quitar filas sin nombre (dato mínimo indispensable), quitar duplicados exactos
dataset_unificado = dataset_unificado.dropna(subset=["nombre"])
dataset_unificado = dataset_unificado.drop_duplicates(subset=["ciudad", "nombre"])

print(f"Dataset unificado: {dataset_unificado.shape[0]} filas")
print(dataset_unificado["ciudad"].value_counts())
print()
dataset_unificado.sample(10)

Dataset unificado: 2666 filas
ciudad
Buenos Aires    1575
Paris            568
New York         523
Name: count, dtype: int64



,ciudad,categoria,nombre,descripcion,ubicacion
430,Buenos Aires,MONUMENTO,HIPOLITO VIEYTES,"Autor: LLANECES, JOSE | Material: GRANITO Y BR...","VIEYTES Y OLAVARRIA, BARRACAS"
2761,New York,Restaurant,TAL BAGELS DELI,Restaurant located in Manhattan,"FIRST AVENUE, Manhattan"
888,Buenos Aires,BUSTO,MAHATMA GHANDI,"Autor: SUTAR, RAM VANJI | Material: BRONCE","DEL LIBERTADOR AV. 3101, PALERMO"
3094,Paris,Histoire,JEP 2026 : Découverte de l'Église Saint-Roch,Redécouvrez l'Arche d'alliance restaurée de l'...,"Église Saint-Roch, 296 rue Saint-Honoré"
538,Buenos Aires,BUSTO,BERNARDINO RIVADAVIA,"Autor: PERLOTI, LUIS | Material: MARMOL","BOLIVAR 1, MONSERRAT"
1529,Buenos Aires,GRUPO ESCULTORICO,LA MUJER Y EL GATO,"Autor: REAL DEL SARTE, M. | Material: MARMOL","CHENAUT, INDALESIO, GRAL. AV. Y CAMPOS, LUIS M..."
3079,Paris,Atelier;Balade urbaine;Nature,Nature à Paris : le programme,"Tous les mois, faîtes-vous plaisir avec des di...",","
2435,New York,Restaurant,ASIA PLAZA CAFÉ,Restaurant located in Bronx,"SOUTHERN BOULEVARD, Bronx"
3478,Paris,S,BOUTEBRIE,S,"15, RUE DE LA PARCHEMINERIE"
1689,Buenos Aires,PLACA Y MONOLITO,ESCUDO NORUEGO,Material: BRONCE Y MAMPOSTERIA,"CIUDAD DE LA PAZ 2175, BELGRANO"


In [10]:
ruta_dataset_final = os.path.join(BASE_PATH, "data", "dataset_unificado.csv")
dataset_unificado.to_csv(ruta_dataset_final, index=False)
print(f"Dataset unificado guardado en: {ruta_dataset_final}")
print(f"Total: {dataset_unificado.shape[0]} filas")

Dataset unificado guardado en: /content/drive/MyDrive/travel_planner_rag/data/dataset_unificado.csv
Total: 2666 filas


Se unificaron 6 datasets heterogéneos (distintos idiomas, columnas y plataformas) en un
esquema común: {ciudad, categoria, nombre, descripcion, ubicacion}.

Decisiones y correcciones aplicadas:
- Filtrado de Paris-Cultura por tags relevantes (Histoire, Expo, Balade, Patrimoine),
  descartando eventos efímeros (conciertos, talleres) sin valor para un itinerario turístico.
  Redujo el dataset de 2843 a 545 filas, corrigiendo además el desbalance entre ciudades.
- Deduplicación de NY-Gastronomía por ID único de establecimiento (el dataset original
  tenía una fila por inspección sanitaria, no por restaurante).
- Manejo correcto de valores nulos (evitar que aparezca el string "nan" o comas/separadores
  sobrantes en los campos de texto combinados).

In [11]:
# Precios de referencia aproximados (USD), basados en guías de costo de viaje 2026.
# Son valores representativos, no precios exactos de un establecimiento puntual.
PRECIOS_REFERENCIA = {
    "Buenos Aires": {
        "desayuno":     {"economico": 4,  "estandar": 10, "premium": 22},
        "almuerzo":     {"economico": 7,  "estandar": 20, "premium": 65},
        "cena":         {"economico": 12, "estandar": 32, "premium": 90},
        "alojamiento":  {"economico": 22, "estandar": 80, "premium": 220},
    },
    "New York": {
        "desayuno":     {"economico": 8,  "estandar": 15, "premium": 32},
        "almuerzo":     {"economico": 12, "estandar": 28, "premium": 65},
        "cena":         {"economico": 20, "estandar": 50, "premium": 150},
        "alojamiento":  {"economico": 75, "estandar": 180, "premium": 450},
    },
    "Paris": {
        "desayuno":     {"economico": 6,  "estandar": 14, "premium": 30},
        "almuerzo":     {"economico": 17, "estandar": 35, "premium": 65},
        "cena":         {"economico": 20, "estandar": 50, "premium": 130},
        "alojamiento":  {"economico": 45, "estandar": 190, "premium": 350},
    },
}

def formatear_tabla_precios(ciudad):
    """Convierte la tabla de precios de una ciudad en texto para el prompt."""
    if ciudad not in PRECIOS_REFERENCIA:
        return ""

    precios = PRECIOS_REFERENCIA[ciudad]
    lineas = [f"Precios de referencia aproximados en {ciudad} (USD):"]
    nombres = {"economico": "Económico", "estandar": "Estándar", "premium": "Premium"}

    for categoria, valores in precios.items():
        fila = ", ".join(f"{nombres[nivel]}: ${monto}" for nivel, monto in valores.items())
        lineas.append(f"- {categoria.capitalize()} → {fila}")

    return "\n".join(lineas)

# Prueba
print(formatear_tabla_precios("Buenos Aires"))

Precios de referencia aproximados en Buenos Aires (USD):
- Desayuno → Económico: $4, Estándar: $10, Premium: $22
- Almuerzo → Económico: $7, Estándar: $20, Premium: $65
- Cena → Económico: $12, Estándar: $32, Premium: $90
- Alojamiento → Económico: $22, Estándar: $80, Premium: $220


### Preparación de los documentos para RAG

In [12]:
dataset_unificado["barrio"] = (
    dataset_unificado["ubicacion"]
    .fillna("")
    .astype(str)
    .str.split(",")
    .str[-1]
    .str.strip()
    .str.upper()
)

In [13]:
dataset_unificado["texto_documento"] = (
    "Ciudad: " + dataset_unificado["ciudad"].fillna("").astype(str) +
    " | Categoría: " + dataset_unificado["categoria"].fillna("").astype(str) +
    " | Nombre: " + dataset_unificado["nombre"].fillna("").astype(str) +
    " | Descripción: " + dataset_unificado["descripcion"].fillna("").astype(str) +
    " | Ubicación: " + dataset_unificado["ubicacion"].fillna("").astype(str)
)

In [14]:
print(dataset_unificado["texto_documento"].iloc[0])

Ciudad: Buenos Aires | Categoría: ESTATUA | Nombre: INTERIORES | Descripción: Autor: ALVAREZ LOMBA, A. | Material: MARMOL DE CORDOBA | Ubicación: BOEDO AV. 883, BOEDO


##  Implementación de RAG

Acá arranca la parte central del proyecto. La idea de fondo es simple: en vez de que el LLM invente lugares de memoria, primero buscamos los más relevantes en nuestra base de datos real, y recién después se los pasamos al modelo para que arme el itinerario. Esto reduce muchísimo el riesgo de alucinaciones.

### RAG - Modelo de embeddings

Usamos un modelo multilingüe (soporta español, inglés y francés — nuestras 3 ciudades) específicamente entrenado para generar embeddings de buena calidad.

In [15]:
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Modelo de embeddings cargado.")
print(f"Dimensión de cada embedding: {embedding_model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings cargado.
Dimensión de cada embedding: 384


/tmp/ipykernel_14302/3701934352.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Dimensión de cada embedding: {embedding_model.get_sentence_embedding_dimension()}")


### RAG - Generar embeddings de todos los documentos

Convertimos cada uno de los 2651 documentos en un vector numérico, que vamos a
almacenar en la base vectorial en el siguiente paso.

In [16]:
textos = dataset_unificado["texto_documento"].tolist()

embeddings = embedding_model.encode(
    textos,
    show_progress_bar=True,
    batch_size=64
)

print(f"Embeddings generados: {embeddings.shape}")

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Embeddings generados: (2666, 384)


###  RAG - Base de datos vectorial (ChromaDB)

Guardamos los embeddings junto con sus documentos originales en ChromaDB, para poder hacer búsquedas de similitud de forma eficiente.

In [17]:
client_chroma = chromadb.PersistentClient(path=os.path.join(BASE_PATH, "chroma_db"))

# Si la colección ya existe, la eliminamos para comenzar limpio
if "lugares_turisticos" in [c.name for c in client_chroma.list_collections()]:
    client_chroma.delete_collection(name="lugares_turisticos")

# Crear colección utilizando similitud coseno
coleccion = client_chroma.create_collection(
    name="lugares_turisticos",
    metadata={"hnsw:space": "cosine"}
)

# Metadata utilizada para los filtros del retrieval
metadata_limpia = (
    dataset_unificado[
        ["ciudad", "categoria", "nombre", "barrio"]
    ]
    .fillna("Sin dato")
    .astype(str)
)

# Indexar documentos, embeddings y metadata
coleccion.add(
    ids=[str(i) for i in dataset_unificado.index],
    embeddings=embeddings.tolist(),
    documents=dataset_unificado["texto_documento"].tolist(),
    metadatas=metadata_limpia.to_dict(orient="records")
)

print(f"Documentos indexados en ChromaDB: {coleccion.count()}")

Documentos indexados en ChromaDB: 2666


### RAG - Retrieval

Construir la función que, dada una consulta del usuario (por ejemplo, sus intereses y destino), devuelva los documentos más relevantes de la base vectorial.

In [18]:
duracion_dias = 3
destino = "Buenos Aires"

intereses = "cultura, arquitectura y gastronomía"
presupuesto = "estandar"
restricciones = "ninguna"

consultas = {
    "cultura": f"""
    Lugares culturales, patrimonio histórico, monumentos,
    esculturas, arquitectura y sitios de interés cultural
    para visitar en {destino}.
    """,

    "atracciones": f"""
    Atracciones turísticas, lugares destacados y sitios
    interesantes para visitar durante un viaje a {destino}.
    """,

    "gastronomia": f"""
    Restaurantes y lugares recomendables para comer
    durante un viaje a {destino}.
    """
}

In [19]:
resultados_por_tipo = {}

for tipo, texto in consultas.items():

    embedding = embedding_model.encode([texto])[0]

    resultados_por_tipo[tipo] = coleccion.query(
        query_embeddings=[embedding.tolist()],
        n_results=5
    )

In [20]:
lugares_unicos = {}

for tipo, resultados in resultados_por_tipo.items():

    for documento, metadata in zip(
        resultados["documents"][0],
        resultados["metadatas"][0]
    ):

        nombre = metadata.get("nombre", "").strip()

        if nombre and nombre not in lugares_unicos:

            lugares_unicos[nombre] = {
                "tipo": tipo,
                "documento": documento,
                "metadata": metadata
            }

print(f"Lugares únicos recuperados: {len(lugares_unicos)}")

Lugares únicos recuperados: 13


In [21]:
for nombre, lugar in lugares_unicos.items():

    print(
        f"[{lugar['tipo'].upper()}] "
        f"{nombre}"
    )

[CULTURA] GLORIAS NAVALES DE LA NACION
[CULTURA] SAN MARTINN DE TOURS
[CULTURA] GIBRAN KHALIL GIBRAN
[CULTURA] DOMINGO MATHEU
[CULTURA] HOMENAJE A LAS VICTIMAS DE LA FIEBRE AM
[ATRACCIONES] HJE.A SAN MARTINN DE TOURS
[ATRACCIONES] HJE.120°ANIV.INAUGURACION PLAZA SOLI
[ATRACCIONES] VICENTE LOPEZ Y PLANES
[GASTRONOMIA] MESÓN NAVARRO
[GASTRONOMIA] GRAN PIZZERÍA JOSÉ
[GASTRONOMIA] ASADOR CRIOLLO LA ESTANCIA
[GASTRONOMIA] CERVECERÍA LÓPEZ
[GASTRONOMIA] ANGUS


### RAG - Construcción del contexto

Vamos a juntar los resultados de las tres búsquedas en un único contexto estructurado.

In [22]:
contexto_rag = ""

for nombre, lugar in lugares_unicos.items():

    contexto_rag += (
        f"[{lugar['tipo'].upper()}]\n"
        f"{lugar['documento']}\n\n"
    )

print("Contexto RAG preparado.")

Contexto RAG preparado.


In [23]:
print(contexto_rag)

[CULTURA]
Ciudad: Buenos Aires | Categoría: MASTIL MONUMENTAL | Nombre: GLORIAS NAVALES DE LA NACION | Descripción: Autor: LEONE, JUAN BAUTISTA | Material: PIEDRA | Ubicación: SALVADORES, CNEL. E IRALA, BOCA

[CULTURA]
Ciudad: Buenos Aires | Categoría: MONUMENTO | Nombre: SAN MARTINN DE TOURS | Descripción: Autor: BUCCI, ERMANDO | Material: BRONCE | Ubicación: ALVEAR AV. 2136, RECOLETA

[CULTURA]
Ciudad: Buenos Aires | Categoría: MONUMENTO | Nombre: GIBRAN KHALIL GIBRAN | Descripción: Material: BRONCE Y GRANITO | Ubicación: PLAZA BARTOLOME MITRE

[CULTURA]
Ciudad: Buenos Aires | Categoría: MONUMENTO | Nombre: DOMINGO MATHEU | Descripción: Autor: ALONSO, MATEO | Material: GRANITO Y BRONCE | Ubicación: ARAOZ DE LAMADRID, GREGORIO, GRAL. Y HERNANDARIAS, BOCA

[CULTURA]
Ciudad: Buenos Aires | Categoría: MONUMENTO | Nombre: HOMENAJE A LAS VICTIMAS DE LA FIEBRE AM | Descripción: Autor: FERRARI, JUAN | Material: MARMOL | Ubicación: CASEROS AV. Y SANTA CRUZ, PARQUE PATRICIOS

[ATRACCIONES]
Ciu

In [24]:
prompt = f"""
Sos un agente de viajes especializado en crear itinerarios personalizados.

Creá un itinerario turístico de {duracion_dias} días para {destino}.

Usá EXCLUSIVAMENTE los lugares incluidos en el CONTEXTO RAG.

PREFERENCIAS DEL VIAJERO
Destino: {destino}
Duración: {duracion_dias} días
Intereses: {intereses}
Presupuesto: {presupuesto}
Restricciones: {restricciones if restricciones else "ninguna"}

CONTEXTO RAG
------------------------------------------
{contexto_rag}
------------------------------------------

PRECIOS DE REFERENCIA
------------------------------------------
{formatear_tabla_precios(destino)}
------------------------------------------

REGLAS

1. Generá exactamente {duracion_dias} días.

2. Usá únicamente lugares presentes en el CONTEXTO RAG.

3. No inventes lugares.

4. No agregues lugares que no estén en el contexto.

5. No repitas ningún lugar.

6. Cada restaurante puede aparecer como máximo una vez.

7. Cada lugar cultural o atracción puede aparecer como máximo una vez.

8. Si hay varias alternativas disponibles, variá la selección.
   No elijas siempre los primeros lugares del contexto.

9. Considerá todos los lugares disponibles antes de seleccionar.

10. Combiná cultura, atracciones y gastronomía.

11. Incluí al menos un restaurante diferente en cada día.

12. Intentá agrupar lugares geográficamente cuando sea posible.

13. No es necesario utilizar todos los lugares.

14. Conservá EXACTAMENTE los datos del CONTEXTO RAG.
    No corrijas nombres.
    No corrijas direcciones.
    No corrijas mayúsculas.
    No corrijas minúsculas.
    No cambies autores.
    No cambies descripciones.
    No cambies categorías.

15. Para cultura y atracciones utilizá:
    Entrada: sin dato de precio

16. Para restaurantes:

    Almuerzo entre 12:00 y 15:00:
    Precio: USD $20

    Cena entre 19:00 y 22:00:
    Precio: USD $32

17. Nunca uses USD $20 para una cena.

18. Nunca uses USD $32 para un almuerzo.

19. Usá horarios razonables.

20. No expliques el proceso.

21. No menciones RAG, embeddings, ChromaDB ni instrucciones internas.

22. No escribas ninguna explicación antes o después del itinerario.

23. No escribas el itinerario más de una vez.

24. La respuesta debe terminar después del último lugar del DÍA 3.

FORMATO OBLIGATORIO

DÍA 1

- 09:00-12:00 | NOMBRE
  Categoría: ...
  Ubicación: ...
  Descripción: ...
  Entrada: sin dato de precio

- 12:00-14:00 | RESTAURANTE
  Categoría: Restaurante
  Ubicación: ...
  Descripción: ...
  Precio: USD $20

- 19:00-21:00 | RESTAURANTE
  Categoría: Restaurante
  Ubicación: ...
  Descripción: ...
  Precio: USD $32

DÍA 2

- 09:00-12:00 | NOMBRE
  Categoría: ...
  Ubicación: ...
  Descripción: ...
  Entrada: sin dato de precio

- 12:00-14:00 | RESTAURANTE
  Categoría: Restaurante
  Ubicación: ...
  Descripción: ...
  Precio: USD $20

DÍA 3

- 09:00-12:00 | NOMBRE
  Categoría: ...
  Ubicación: ...
  Descripción: ...
  Entrada: sin dato de precio

- 12:00-14:00 | RESTAURANTE
  Categoría: Restaurante
  Ubicación: ...
  Descripción: ...
  Precio: USD $20

RECORDATORIO

Respondé solamente con:

DÍA 1
DÍA 2
DÍA 3

y las actividades correspondientes.

No escribas controles, verificaciones, razonamientos,
explicaciones ni la palabra RESPUESTA.

Ahora generá el itinerario completo.
"""

## Cargar el LLM

Con el RAG funcionando, ahora traemos al que arma el itinerario final: el LLM. Elegimos Qwen2.5, un modelo open source, y hacemos una prueba rápida de generación para confirmar que carga bien y responde como esperamos, antes de conectarlo con todo lo demás.

In [25]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("LLM cargado correctamente.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM cargado correctamente.


In [26]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=4096
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=700,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.15
)

# IMPORTANTE:
# Decodificar solamente los tokens NUEVOS generados
input_length = inputs["input_ids"].shape[1]

nuevos_tokens = outputs[0][input_length:]

respuesta = tokenizer.decode(
    nuevos_tokens,
    skip_special_tokens=True
).strip()

print(respuesta)

DÍA 1
- 09:00-12:00 | DOMINGO MATHEU 
  Categoría: Monumento  
  Ubicación: Araoz de Lamadrid, Gregorio, Gral. y Hernandarias, Boca  
  Descripción: Autor: Alonso, Mateo  
  Material: Granito y Bronce  

- 12:00-14:00 | MESÓN NAVARRO  
  Categoría: Restaurante   
  Ubicación: Lavalle 170  
  Descripción: Restaurante en el barrio de San Nicolás  

- 19:00-21:00 | ANGUS  
  Categoría: Restaurante   
  Ubicación: Avalos 2184  
  Descripción: Restaurante en el barrio de Villa Urquiza   

DÍA 2
- 09:00-12:00 | HUELTAS DE SAN MARTINN DE TOURS 
  Categoría: Placa y Monumento  
  Ubicación: Av. General Paz 4646  
  Descripción: Autor: D. G. O. M. y M. O. A.  
  Material: Bronze y Mamposteria  

- 12:00-14:00 | GRAN PIZZERÍA JOSÉ  
  Categoría: Restaurante   
  Ubicación: Av. San Martín 6915  
  Descripción: Restaurante en el barrio de Villa Devoto  

- 19:00-21:00 | CERVECERÍA LÓPEZ  
  Categoría: Restaurante   
  Ubicación: Alvarez Thomas 2136  
  Descripción: Restaurante en el barrio de Vill

## Interfaz

Y para cerrar, la parte que se ve: armamos una interfaz simple con Streamlit para que cualquiera —no solo alguien que sepa Python— pueda usar el planificador. Elegís destino, días, intereses, presupuesto y restricciones, tocás un botón, y el sistema hace todo el trabajo de RAG + LLM por detrás para devolverte el itinerario armado.

In [27]:
%%writefile app.py
import os

import chromadb
import streamlit as st
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer


BASE_PATH = os.environ.get("TRAVEL_PLANNER_BASE_PATH", "/content/drive/MyDrive/travel_planner_rag")
CHROMA_PATH = os.path.join(BASE_PATH, "chroma_db")
COLLECTION_NAME = "lugares_turisticos"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

PRECIOS_REFERENCIA = {
    "Buenos Aires": {
        "desayuno": {"economico": 4, "estandar": 10, "premium": 22},
        "almuerzo": {"economico": 7, "estandar": 20, "premium": 65},
        "cena": {"economico": 12, "estandar": 32, "premium": 90},
        "alojamiento": {"economico": 22, "estandar": 80, "premium": 220},
    },
    "New York": {
        "desayuno": {"economico": 8, "estandar": 15, "premium": 32},
        "almuerzo": {"economico": 12, "estandar": 28, "premium": 65},
        "cena": {"economico": 20, "estandar": 50, "premium": 150},
        "alojamiento": {"economico": 75, "estandar": 180, "premium": 450},
    },
    "Paris": {
        "desayuno": {"economico": 6, "estandar": 14, "premium": 30},
        "almuerzo": {"economico": 17, "estandar": 35, "premium": 65},
        "cena": {"economico": 20, "estandar": 50, "premium": 130},
        "alojamiento": {"economico": 45, "estandar": 190, "premium": 350},
    },
}


def formatear_tabla_precios(ciudad):
    etiquetas = {"economico": "Económico", "estandar": "Estándar", "premium": "Premium"}
    lineas = [f"Precios de referencia aproximados en {ciudad} (USD):"]
    for categoria, valores in PRECIOS_REFERENCIA[ciudad].items():
        detalle = ", ".join(f"{etiquetas[nivel]}: ${monto}" for nivel, monto in valores.items())
        lineas.append(f"- {categoria.capitalize()}: {detalle}")
    return "\n".join(lineas)


@st.cache_resource(show_spinner="Cargando el modelo y la base de conocimiento...")
def cargar_recursos():
    if not os.path.exists(CHROMA_PATH):
        raise FileNotFoundError("No se encontró la base vectorial persistida.")

    cliente = chromadb.PersistentClient(path=CHROMA_PATH)
    coleccion = cliente.get_collection(COLLECTION_NAME)
    if coleccion.count() == 0:
        raise RuntimeError("La colección de lugares turísticos está vacía.")

    embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        device_map="auto",
    )
    model.eval()
    return coleccion, embeddings, tokenizer, model


def recuperar_contexto(coleccion, embeddings, destino, intereses):
    consultas = {
        "cultura": f"Lugares culturales, patrimonio, arquitectura y museos para visitar en {destino}. Intereses: {intereses}.",
        "atracciones": f"Atracciones turísticas y sitios destacados para visitar en {destino}. Intereses: {intereses}.",
        "gastronomia": f"Restaurantes y gastronomía recomendables para comer en {destino}.",
    }
    lugares = {}
    for tipo, consulta in consultas.items():
        resultado = coleccion.query(
            query_embeddings=[embeddings.encode(consulta).tolist()],
            n_results=5,
            where={"ciudad": destino},
        )
        documentos = resultado.get("documents", [[]])[0]
        metadatos = resultado.get("metadatas", [[]])[0]
        for documento, metadata in zip(documentos, metadatos):
            nombre = metadata.get("nombre", "").strip()
            if nombre and nombre not in lugares:
                lugares[nombre] = {"tipo": tipo, "documento": documento}

    if not lugares:
        raise ValueError(f"No hay lugares indexados para {destino}.")

    return "\n\n".join(
        f"[{lugar['tipo'].upper()}]\n{lugar['documento']}"
        for lugar in lugares.values()
    )


def construir_prompt(destino, duracion, intereses, presupuesto, restricciones, contexto):
    precios = PRECIOS_REFERENCIA[destino]
    precio_almuerzo = precios["almuerzo"][presupuesto]
    precio_cena = precios["cena"][presupuesto]
    return f"""Sos un agente de viajes especializado en crear itinerarios personalizados.

Creá exactamente un itinerario turístico de {duracion} días para {destino}.
Usá exclusivamente los lugares incluidos en el CONTEXTO RAG.

PREFERENCIAS DEL VIAJERO
- Destino: {destino}
- Duración: {duracion} días
- Intereses: {intereses}
- Presupuesto: {presupuesto}
- Restricciones: {restricciones or "ninguna"}

CONTEXTO RAG
------------------------------------------
{contexto}
------------------------------------------

PRECIOS DE REFERENCIA
------------------------------------------
{formatear_tabla_precios(destino)}
------------------------------------------

REGLAS OBLIGATORIAS
1. Usá solo lugares del contexto y no repitas ninguno.
2. Priorizá los intereses indicados y combiná cultura, atracciones y gastronomía cuando haya opciones.
3. Incluí al menos un restaurante distinto por día.
4. Para cultura y atracciones escribí exactamente: Entrada: sin dato de precio.
5. Para un almuerzo escribí exactamente: Precio: USD ${precio_almuerzo}.
6. Para una cena escribí exactamente: Precio: USD ${precio_cena}.
7. No inventes datos, no expliques el proceso y no menciones RAG, embeddings ni ChromaDB.
8. Respondé únicamente con el itinerario final, sin repetirlo.

FORMATO
DÍA 1

- HH:MM-HH:MM | NOMBRE
  Categoría: ...
  Ubicación: ...
  Descripción: ...
  Entrada: sin dato de precio

- 12:00-14:00 | RESTAURANTE
  Categoría: Restaurante
  Ubicación: ...
  Descripción: ...
  Precio: USD ${precio_almuerzo}

Repetí este formato hasta completar DÍA {duracion}."""


def generar_itinerario(tokenizer, model, prompt):
    mensajes = [
        {"role": "system", "content": "Sos un asistente de viajes preciso y respetás todas las restricciones del usuario."},
        {"role": "user", "content": prompt},
    ]
    texto_entrada = tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(texto_entrada, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1800,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    nuevos_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(nuevos_tokens, skip_special_tokens=True).strip()


st.set_page_config(page_title="Planificador de viajes", page_icon="✈️", layout="centered")
st.title("Planificador de viajes")
st.write("Creá un itinerario personalizado según tus preferencias.")

with st.form("preferencias_viaje"):
    destino = st.selectbox("Destino", list(PRECIOS_REFERENCIA))
    duracion_dias = st.number_input("Duración (días)", min_value=1, max_value=7, value=3, step=1)
    intereses = st.selectbox(
        "Intereses",
        [
            "cultura", "arquitectura", "gastronomía", "cultura y arquitectura",
            "cultura y gastronomía", "arquitectura y gastronomía",
            "cultura, arquitectura y gastronomía",
        ],
    )
    presupuesto = st.selectbox("Presupuesto", ["economico", "estandar", "premium"], format_func=lambda valor: valor.capitalize())
    restricciones = st.text_input("Restricciones (opcional)", placeholder="Ej.: movilidad reducida, vegetariano")
    generar = st.form_submit_button("Generar itinerario", use_container_width=True)

if generar:
    try:
        with st.spinner("Generando itinerario..."):
            coleccion, embeddings, tokenizer, model = cargar_recursos()
            contexto = recuperar_contexto(coleccion, embeddings, destino, intereses)
            prompt = construir_prompt(destino, int(duracion_dias), intereses, presupuesto, restricciones, contexto)
            itinerario = generar_itinerario(tokenizer, model, prompt)
        st.subheader("Tu itinerario")
        st.markdown(itinerario)
    except Exception as error:
        st.error(f"No se pudo generar el itinerario: {error}")
        st.info("Antes de abrir la interfaz, ejecutá nuevamente la celda que indexa los documentos en ChromaDB.")


Writing app.py


In [28]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &> /content/streamlit.log &


In [29]:
!cat /content/streamlit.log


In [30]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [31]:
import subprocess
import time
import gc
import torch

# Detener cualquier Streamlit anterior
subprocess.run(
    ["pkill", "-9", "-f", "streamlit"],
    capture_output=True
)

time.sleep(2)

# Liberar memoria
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Iniciar Streamlit
log_file = open("/content/streamlit.log", "w")

proceso = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
        "--browser.gatherUsageStats=false",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("Esperando a que Streamlit inicie...")
time.sleep(12)

# Obtener URL pública de Colab
from google.colab.output import eval_js

url = eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print()
print("==========================================")
print("🚀 INTERFAZ STREAMLIT")
print("==========================================")
print(url)

Esperando a que Streamlit inicie...

🚀 INTERFAZ STREAMLIT
https://8501-gpu-t4-s-kkb-usw1b2-1vrqb8cz4uv2m-b.us-west1-2.prod.colab.dev
